# Visualize Results
Notebook to load saved .npz results and plot simple diagnostics.

In [ ]:
import numpy as np
from sf_recon.utils.io import load_npz
print('Ready to load results: call load_npz with path')

In [1]:
# ==============================================================================
# Velocity fields visualization
# Line 1: vn-GT、vs-GT
# Line 2: vn-Recon、vs-Recon
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Domain parameters
Lx, Ly = 0.0004, 0.0016

# Load result
with np.load('../data/simulation/cf_v0_recon.npz', allow_pickle=True) as data:
    # vn fields
    v_gt_speed = data.get('v_gt_speed')
    v_recon_speed = data.get('v_recon_speed')
    v_gt_u = data.get('v_gt_u')
    v_gt_v = data.get('v_gt_v')
    v_recon_u = data.get('v_recon_u')
    v_recon_v = data.get('v_recon_v')
    # vs fields
    vs_gt_speed = data.get('vs_gt_speed')
    vs_recon_speed = data.get('vs_recon_speed')
    vs_gt_u = data.get('vs_gt_u')
    vs_gt_v = data.get('vs_gt_v')
    vs_recon_u = data.get('vs_recon_u')
    vs_recon_v = data.get('vs_recon_v')
    gt = data.get('gt')
    recon = data.get('recon')

def as_time_array(x):
    if x is None:
        return None
    try:
        a = np.array(x)
    except Exception:
        try:
            a = np.array(list(x))
        except Exception:
            return None
    if a.dtype == object:
        try:
            stacked = [np.asarray(e, dtype=float) for e in a]
            a = np.stack(stacked)
        except Exception:
            return None
    try:
        return a.astype(float)
    except Exception:
        return None

v_gt_speed = as_time_array(v_gt_speed)
v_recon_speed = as_time_array(v_recon_speed)
v_gt_u = as_time_array(v_gt_u)
v_gt_v = as_time_array(v_gt_v)
v_recon_u = as_time_array(v_recon_u)
v_recon_v = as_time_array(v_recon_v)
vs_gt_speed = as_time_array(vs_gt_speed)
vs_recon_speed = as_time_array(vs_recon_speed)
vs_gt_u = as_time_array(vs_gt_u)
vs_gt_v = as_time_array(vs_gt_v)
vs_recon_u = as_time_array(vs_recon_u)
vs_recon_v = as_time_array(vs_recon_v)


# Determine the number of frames
T_candidates = []
for arr in (v_gt_speed, v_recon_speed, v_gt_u, v_recon_u, vs_gt_speed, vs_recon_speed, gt, recon):
    if arr is not None:
        try:
            T_candidates.append(int(np.asarray(arr).shape[0]))
        except Exception:
            pass
T = max(T_candidates) if T_candidates else 1

# Grid Building
X = Y = None
u_shape = None
for arr in (v_gt_u, v_recon_u, vs_gt_u, vs_recon_u):
    if arr is not None:
        u_shape = arr.shape
        break
if u_shape is not None and len(u_shape) >= 3:
    H, W = u_shape[1], u_shape[2]
    x = np.linspace(0, Lx, W)
    y = np.linspace(0, Ly, H)
    X, Y = np.meshgrid(x, y)

# Velocity range (if available)
if v_gt_speed is not None:
    vmin_gt, vmax_gt = np.nanmin(v_gt_speed), np.nanmax(v_gt_speed)
else:
    vmin_gt = vmax_gt = None
if v_recon_speed is not None:
    vmin_re, vmax_re = np.nanmin(v_recon_speed), np.nanmax(v_recon_speed)
else:
    vmin_re = vmax_re = None
if vs_gt_speed is not None:
    vs_vmin_gt, vs_vmax_gt = np.nanmin(vs_gt_speed), np.nanmax(vs_gt_speed)
else:
    vs_vmin_gt = vs_vmax_gt = None
if vs_recon_speed is not None:
    vs_vmin_re, vs_vmax_re = np.nanmin(vs_recon_speed), np.nanmax(vs_recon_speed)
else:
    vs_vmin_re = vs_vmax_re = None

# Create figure and axes
fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
fig.subplots_adjust(wspace=0.25, hspace=0.25)

# Titles
titles = ['vn-GT', 'vs-GT', 'vn-Recon', 'vs-Recon']

# Animation frame drawing
def draw_frame(ax, i, speed_arr, u_arr, v_arr, pts, title, vmin=None, vmax=None):
    ax.clear()
    ax.set_xlim(0, Lx)
    ax.set_ylim(0, Ly)
    ax.set_aspect('equal')
    # Velocity cloud map
    if speed_arr is not None:
        try:
            data = np.asarray(speed_arr[i], dtype=float)
            im = ax.imshow(data, origin='lower', extent=[0, Lx, 0, Ly], cmap='Oranges', vmin=vmin, vmax=vmax)
            if not hasattr(ax, '_has_cbar'):
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('speed')
                ax._has_cbar = True
        except Exception:
            pass
    else:
        ax.set_facecolor('whitesmoke')
    # Streamlines
    if u_arr is not None and v_arr is not None and X is not None:
        try:
            u = np.asarray(u_arr[i], dtype=float)
            v = np.asarray(v_arr[i], dtype=float)
            ax.streamplot(X, Y, u, v, color='k', linewidth=0.6, arrowsize=1.0, density=1.0)
        except Exception:
            pass
    # Particles
    if pts is not None:
        try:
            p = np.asarray(pts[i])
            ax.scatter(p[:,0], p[:,1], s=20, facecolors='white', edgecolors='black', linewidths=0.4, zorder=10)
        except Exception:
            pass
    ax.set_title(f'{title} t={i}')

# Update: use vs fields for the 2nd and 4th panels
def update(i):
    draw_frame(axs[0,0], i, v_gt_speed, v_gt_u, v_gt_v, gt, titles[0], vmin_gt, vmax_gt)
    draw_frame(axs[0,1], i, vs_gt_speed, vs_gt_u, vs_gt_v, gt, titles[1], vs_vmin_gt, vs_vmax_gt)
    draw_frame(axs[1,0], i, v_recon_speed, v_recon_u, v_recon_v, recon, titles[2], vmin_re, vmax_re)
    draw_frame(axs[1,1], i, vs_recon_speed, vs_recon_u, vs_recon_v, recon, titles[3], vs_vmin_re, vs_vmax_re)
    return []

num_frames = min(20, T)
anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=200, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

C:\Users\XU_li\AppData\Local\Temp\ipykernel_9788\3654175997.py:107: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(wspace=0.25, hspace=0.25)


In [ ]:
# ==============================================================================
# Temperature Visualization
# L: GT，R: Recon
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Load temperature data
with np.load('../data/simulation/cf_v0_recon.npz', allow_pickle=True) as data:
    t_gt = data.get('t_gt')
    t_recon = data.get('t_recon')

try:
    as_time_array
except NameError:
    def as_time_array(x):
        if x is None:
            return None
        try:
            a = np.array(x)
        except Exception:
            try:
                a = np.array(list(x))
            except Exception:
                return None
        if a.dtype == object:
            try:
                a = np.stack([np.asarray(e, dtype=float) for e in a])
            except Exception:
                return None
        try:
            return a.astype(float)
        except Exception:
            return None

t_gt = as_time_array(t_gt)
t_recon = as_time_array(t_recon)

if t_gt is None and t_recon is None:
    print('No temperature data available in NPZ.')
else:
    sample = t_gt if t_gt is not None else t_recon
    T = int(sample.shape[0])
    H, W = sample.shape[1], sample.shape[2]
    Lx, Ly = 0.0004, 0.0016

    # temperature range
    if t_gt is not None:
        gt_arr = np.asarray(t_gt, dtype=float)
        if np.any(np.isfinite(gt_arr)):
            tmin = float(np.round(np.nanmin(gt_arr), 6))
            tmax = float(np.round(np.nanmax(gt_arr), 6))
        else:
            tmin, tmax = 0.0, 1.0
    else:
        tmin, tmax = 0.0, 1.0

    fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    cmap = plt.cm.inferno
    norm = plt.Normalize(vmin=tmin, vmax=tmax)
    im0 = axs[0].imshow(t_gt[0] if t_gt is not None else np.zeros((H, W)), origin='lower', extent=[0, Lx, 0, Ly], cmap=cmap, norm=norm)
    im1 = axs[1].imshow(t_recon[0] if t_recon is not None else np.zeros((H, W)), origin='lower', extent=[0, Lx, 0, Ly], cmap=cmap, norm=norm)
    axs[0].set_title('Temperature GT')
    axs[1].set_title('Temperature Recon')
    for ax in axs:
        ax.set_xlim(0, Lx)
        ax.set_ylim(0, Ly)
        ax.set_aspect('equal')

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axs.ravel().tolist(), fraction=0.046, pad=0.04)
    try:
        cbar.set_ticks([tmin, tmax])
        cbar.set_ticklabels([f"{tmin:.6f}", f"{tmax:.6f}"])
    except Exception:
        pass
    cbar.set_label('Temperature')

    def update_temp(i):
        if t_gt is not None:
            im0.set_data(np.asarray(t_gt[i], dtype=float))
        if t_recon is not None:
            im1.set_data(np.asarray(t_recon[i], dtype=float))
        axs[0].set_title(f'Temperature GT t={i}')
        axs[1].set_title(f'Temperature Recon t={i}')
        return im0, im1

    num_frames = min(20, T)
    anim = animation.FuncAnimation(fig, update_temp, frames=num_frames, interval=300, blit=False)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))